In [1]:
# 忽视警告，这个库是内置的，不需要安装
from pathlib import Path
import sys
import warnings

warnings.filterwarnings('ignore')

NOTEBOOK_DIR_CANDIDATES = [
    Path.cwd(),
    Path.cwd() / 'lgbm_quicktest',
    Path.cwd().parent / 'lgbm_quicktest',
]
NOTEBOOK_DIR = next(
    (
        path.resolve()
        for path in NOTEBOOK_DIR_CANDIDATES
        if (path / '质谱数据汇总_处理后后后2.csv').exists()
    ),
    None,
)
if NOTEBOOK_DIR is None:
    raise FileNotFoundError('找不到 lgbm_quicktest/质谱数据汇总_处理后后后2.csv')

PROJECT_ROOT = NOTEBOOK_DIR.parent

from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score


In [2]:
import pandas as pd

data_path = NOTEBOOK_DIR / '质谱数据汇总_处理后后后2.csv'

# 使用 pandas 的 read_csv 函数读取 CSV 文件
data = pd.read_csv(data_path)

# # 随机打乱数据集，这一步的操作是可选的，类似于把练习题不断更改顺序，防止电脑学习到固定的顺序，这一步是可选的
random_seed = 42
data = data.sample(frac=1, random_state=random_seed).reset_index(drop=True)

# 显示前几行数据，以确认数据已正确加载

data.head(10)


,毒性,PN,ID,BP,BPP,MaxM,MaxMP,MinM,MM,MSD,IM,ISD,RETENTION_TIME,COLLISION_ENERGY,PRECURSOR_TYPE
2856,0,6,166.500000,262.06500,18.01410,264.06750,1.00070,195.07650,237.402350,30.333003,9.092667e+03,1.550308e+04,7.186,20.0,0.0
4397,0,2,499.500000,73.00000,1.00000,73.00000,1.00000,72.00000,72.500000,0.500000,5.940400e+04,4.456900e+04,NaN,10.0,1.0
7689,0,1,999.000000,231.02170,0.00000,231.02170,0.00000,231.02170,231.021700,0.000000,4.830730e+04,0.000000e+00,12.7,15.0,0.0
15714,1,54,18.500000,120.08130,1.00840,337.18400,1.00300,79.05390,141.907941,62.798725,1.240778e+03,2.282639e+03,NaN,50.0,1.0
9004,0,14,71.357143,80.04950,1.00790,118.06540,12.00030,51.02290,78.690321,19.844384,2.967256e+06,6.250563e+06,0.962,150.0,1.0
1034,0,5,199.800000,271.06018,118.04234,271.06018,118.04234,91.05567,155.841806,61.564909,2.300000e+02,3.860658e+02,5.891117,6.0,1.0
11585,1,85,11.752941,205.07730,0.97130,320.14030,18.01060,56.04930,187.073838,43.955155,1.688978e+04,3.108147e+04,4.239,80.0,1.0
13449,1,8,124.875000,266.98490,1.99190,283.98760,17.00270,73.02830,217.628000,62.187735,9.227507e+04,1.667297e+05,14.8,30.0,1.0
8863,0,5,199.800000,106.10000,26.80000,107.00000,0.90000,76.90000,89.540000,13.912670,2.786141e+06,3.823010e+06,NaN,40.0,1.0
1355,0,103,9.699029,122.04140,1.00740,205.04060,10.99890,42.04620,123.467771,33.679021,1.339806e+00,1.185621e+00,5.185,110.0,1.0


In [3]:
import pandas as pd

retention_time_raw = data.loc[:, 'RETENTION_TIME'].copy()
retention_time_numeric = pd.to_numeric(retention_time_raw, errors='coerce')
data.loc[:, 'RETENTION_TIME'] = retention_time_numeric.astype('float64')

print("转换后 RETENTION_TIME 的示例值：")
print(data.loc[:, 'RETENTION_TIME'].head())

# 检查非数值数据（如字符串、空值等）
non_numeric = pd.isna(retention_time_numeric)
print("\n非数值数据数量：", int(non_numeric.sum()))
print("非数值数据示例：")
print(retention_time_raw.loc[non_numeric].unique())  # 查看原始非数值内容


转换后 RETENTION_TIME 的示例值：
2856      7.186
4397        NaN
7689     12.700
15714       NaN
9004      0.962
Name: RETENTION_TIME, dtype: float64

非数值数据数量： 3266
非数值数据示例：
<ArrowStringArray>
[nan, '17.9  and 18.5']
Length: 2, dtype: str


In [4]:
from sklearn.model_selection import train_test_split  # 用于将数据集划分为训练集和测试集
from sklearn.preprocessing import StandardScaler  # 用于特征标准化

# 分离特征数据和标签数据
X = data.drop(columns=['毒性']).copy()  # 特征数据
y = data.loc[:, '毒性'].copy()  # 标签数据

y


2856     0
4397     0
7689     0
15714    1
9004     0
        ..
11284    1
11964    1
5390     0
860      0
15795    1
Name: 毒性, Length: 16806, dtype: int64

In [5]:
# 1. 先划分数据集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = X_train.copy()
X_test = X_test.copy()
y_train = y_train.copy()
y_test = y_test.copy()

# 2. 对数值列进行缺失值填补，使用训练集众数，避免数据泄露
numeric_columns = X_train.select_dtypes(include=['number']).columns.tolist()
numeric_impute_values = {}
for column in numeric_columns:
    series = X_train.loc[:, column]
    if series.notna().any():
        mode = series.mode(dropna=True)
        fill_value = mode.iloc[0] if not mode.empty else 0
    else:
        fill_value = 0
    numeric_impute_values[column] = fill_value

if numeric_columns:
    X_train.loc[:, numeric_columns] = X_train.loc[:, numeric_columns].fillna(value=numeric_impute_values)
    X_test.loc[:, numeric_columns] = X_test.loc[:, numeric_columns].fillna(value=numeric_impute_values)

# 3. 分离连续特征和离散特征
continuous_features_train = X_train.select_dtypes(include=['number']).columns.tolist()
continuous_features_test = X_test.select_dtypes(include=['number']).columns.tolist()
discrete_features_train = X_train.select_dtypes(exclude=['number']).columns.tolist()
discrete_features_test = X_test.select_dtypes(exclude=['number']).columns.tolist()

# 4. 连续特征标准化
scaler = StandardScaler()
X_train_continuous = scaler.fit_transform(X_train.loc[:, continuous_features_train])
X_test_continuous = scaler.transform(X_test.loc[:, continuous_features_test])

# 5. 组合处理后的连续特征和离散特征
X_train_processed = pd.DataFrame(X_train_continuous, columns=continuous_features_train, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_continuous, columns=continuous_features_test, index=X_test.index)

# 如果存在离散特征，需要将它们添加回来
if discrete_features_train:
    X_train_processed = pd.concat([X_train_processed, X_train.loc[:, discrete_features_train]], axis=1)
if discrete_features_test:
    X_test_processed = pd.concat([X_test_processed, X_test.loc[:, discrete_features_test]], axis=1)

# 将处理后的数据重新赋值
X_train = X_train_processed
X_test = X_test_processed
training_feature_columns = X_train.columns.tolist()
y_train_name = y_train.name or '毒性'
y_test_name = y_test.name or y_train_name

# 6. 在 SMOTE 前做健壮性检查
train_nan_columns = X_train.columns[X_train.isna().any()].tolist()
test_nan_columns = X_test.columns[X_test.isna().any()].tolist()
if train_nan_columns or test_nan_columns:
    nan_columns = train_nan_columns + [column for column in test_nan_columns if column not in train_nan_columns]
    raise ValueError(f'预处理后仍存在 NaN，无法执行 SMOTE。涉及列: {nan_columns}')

non_numeric_columns = X_train.select_dtypes(exclude=['number']).columns.tolist()
if non_numeric_columns:
    raise TypeError(f'SMOTE 前仍存在非数值列，请先编码: {non_numeric_columns}')

# 7. 对训练集进行过采样（SMOTE）
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
X_train = pd.DataFrame(X_train_resampled, columns=training_feature_columns)
y_train = pd.Series(y_train_resampled, name=y_train_name)
y_test = pd.Series(y_test, name=y_test_name)

# 检查过采样后的类别分布
print('过采样后的训练集形状:', X_train.shape)
print('过采样后的标签分布:', y_train.value_counts())


ValueError: Input X contains NaN.
SMOTE does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values

xgboost不调参0.2s，调参跑了17s

## LightGBM：953 √

In [ ]:
from importlib import import_module
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix
from sklearn.model_selection import GridSearchCV

# 如果内核从项目根目录启动，先移除本地 lightgbm 目录对官方包导入的遮蔽。
current_dir = Path.cwd().resolve()
if (current_dir / 'lightgbm').is_dir():
    sys.path = [
        path
        for path in sys.path
        if Path(path or current_dir).resolve() != current_dir
    ]

lgb = import_module('lightgbm')

# 定义 LightGBM 模型
lgb_model = lgb.LGBMClassifier(random_state=42)

# 定义参数网格
param_grid = {
    'num_leaves': [127],  # 每棵树的最大叶子节点数
    'learning_rate': [0.1],  # 学习率
    'n_estimators': [500],  # 树的数量
    'max_depth': [-1],  # 树的最大深度，-1 表示不限制
    'subsample': [0.7],  # 样本采样比例
    'colsample_bytree': [0.7]  # 特征采样比例
}

# 初始化网格搜索
grid_search = GridSearchCV(
    estimator=lgb_model,  # 模型
    param_grid=param_grid,  # 参数网格
    cv=5,  # 五折交叉验证
    scoring='roc_auc',  # 使用 AUC 作为评估指标
    n_jobs=-1,  # 使用所有可用的CPU核心
    verbose=1
)

# 在训练集上执行网格搜索
grid_search.fit(X_train, y_train)

# 输出最佳参数
print('最佳参数组合:', grid_search.best_params_)
print('最佳验证集AUC:', grid_search.best_score_)

# 使用最佳模型预测测试集
y_pred_proba = grid_search.best_estimator_.predict_proba(X_test)[:, 1]
test_auc = roc_auc_score(y_test, y_pred_proba)
print('测试集AUC:', test_auc)

# 可选：也输出测试集的准确率和F1分数
y_pred = grid_search.best_estimator_.predict(X_test)
test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)
print('测试集准确率:', test_accuracy)
print('测试集F1分数:', test_f1)


In [ ]:
# 使用最佳参数训练模型
best_lgb = grid_search.best_estimator_

# ============ 训练集评估 ============
y_train_pred = best_lgb.predict(X_train)
y_train_proba = best_lgb.predict_proba(X_train)[:, 1]

# 计算训练集指标
train_metrics = {
    'AUC': roc_auc_score(y_train, y_train_proba),
    'Accuracy': accuracy_score(y_train, y_train_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_train, y_train_pred),
    'Precision': precision_score(y_train, y_train_pred),
    'Recall': recall_score(y_train, y_train_pred),
    'F1': f1_score(y_train, y_train_pred)
}

# 计算Specificity
tn, fp, fn, tp = confusion_matrix(y_train, y_train_pred).ravel()
train_metrics['Specificity'] = tn / (tn + fp)

# ============ 测试集评估 ============
y_test_pred = best_lgb.predict(X_test)
y_test_proba = best_lgb.predict_proba(X_test)[:, 1]

# 计算测试集指标
test_metrics = {
    'AUC': roc_auc_score(y_test, y_test_proba),
    'Accuracy': accuracy_score(y_test, y_test_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_test_pred),
    'Precision': precision_score(y_test, y_test_pred),
    'Recall': recall_score(y_test, y_test_pred),
    'F1': f1_score(y_test, y_test_pred)
}

# 计算Specificity
tn, fp, fn, tp = confusion_matrix(y_test, y_test_pred).ravel()
test_metrics['Specificity'] = tn / (tn + fp)

# ============ 打印结果 ============
print("\n=== 训练集性能 ===")
for metric, value in train_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

print("\n=== 测试集性能 ===")
for metric, value in test_metrics.items():
    print(f"{metric:<18}: {value:.4f}")

# 打印混淆矩阵
print("\n测试集混淆矩阵:")
print(confusion_matrix(y_test, y_test_pred))

# 特征重要性可视化（可选）
lgb.plot_importance(best_lgb, max_num_features=20)
plt.tight_layout()
plt.show()

In [ ]:
import json
import joblib

def _to_python_scalar(value):
    if hasattr(value, 'item'):
        try:
            return value.item()
        except Exception:
            pass
    return value

artifact_dir = PROJECT_ROOT / 'models' / 'lgbm'
artifact_dir.mkdir(parents=True, exist_ok=True)

export_numeric_impute_values = {
    column: _to_python_scalar(value)
    for column, value in numeric_impute_values.items()
}
feature_order = list(getattr(best_lgb, 'feature_name_', training_feature_columns))
raw_input_columns = X.columns.tolist()
continuous_features = list(continuous_features_train)
discrete_features = list(discrete_features_train)
retention_time_cast_columns = ['RETENTION_TIME'] if 'RETENTION_TIME' in raw_input_columns else []

preprocessor_bundle = {
    'numeric_impute_values': export_numeric_impute_values,
    'continuous_features': continuous_features,
    'discrete_features': discrete_features,
    'feature_order': feature_order,
    'raw_input_columns': raw_input_columns,
    'raw_input_dtypes': {column: str(dtype) for column, dtype in X.dtypes.items()},
    'processed_feature_dtypes': {column: str(dtype) for column, dtype in X_test.loc[:, feature_order].dtypes.items()},
    'retention_time_cast_columns': retention_time_cast_columns,
    'target_column': '毒性',
    'test_size': 0.2,
    'random_state': 42,
    'scaler': scaler,
}

model_joblib_path = artifact_dir / 'lightgbm_model.joblib'
model_txt_path = artifact_dir / 'lightgbm_model.txt'
preprocessor_path = artifact_dir / 'lightgbm_preprocessor.joblib'
raw_template_path = artifact_dir / 'raw_input_template.csv'
processed_template_path = artifact_dir / 'processed_input_template.csv'
manifest_path = artifact_dir / 'inference_assets.json'

joblib.dump(best_lgb, model_joblib_path)
best_lgb.booster_.save_model(str(model_txt_path))
joblib.dump(preprocessor_bundle, preprocessor_path)

X.loc[:, raw_input_columns].head(0).to_csv(raw_template_path, index=False, encoding='utf-8-sig')
X_test.loc[:, feature_order].head(0).to_csv(processed_template_path, index=False, encoding='utf-8-sig')

manifest = {
    'artifact_dir': str(artifact_dir.resolve()),
    'model_joblib_path': str(model_joblib_path.resolve()),
    'model_txt_path': str(model_txt_path.resolve()),
    'preprocessor_path': str(preprocessor_path.resolve()),
    'raw_input_template_path': str(raw_template_path.resolve()),
    'processed_input_template_path': str(processed_template_path.resolve()),
    'training_data_path': str(data_path.resolve()),
    'target_column': '毒性',
    'retention_time_cast_columns': retention_time_cast_columns,
    'continuous_features': continuous_features,
    'discrete_features': discrete_features,
    'feature_order': feature_order,
    'numeric_impute_values': export_numeric_impute_values,
}
with manifest_path.open('w', encoding='utf-8') as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print('LightGBM 推理资产已保存:')
for path in [model_joblib_path, model_txt_path, preprocessor_path, raw_template_path, processed_template_path, manifest_path]:
    print(f'- {path.resolve()}')
